In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from pathlib import Path
import copy
import json
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import nn

from scipy.integrate import solve_ivp
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

In [3]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Proyecto_PINN_HRF"
)

SCENARIO = "motor_lr_m1_left"
SNR_NAME = "snr_5"
REPLICATE = 1

NOISY_FILE = (
    PROJECT_ROOT
    / "data"
    / "synthetic"
    / "noisy"
    / SCENARIO
    / SNR_NAME
    / (
        f"{SCENARIO}_"
        f"{SNR_NAME}_"
        f"rep_{REPLICATE:02d}.csv"
    )
)

EVENT_FILE = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "100206"
    / "MNINonLinear"
    / "Results"
    / "tfMRI_MOTOR_LR"
    / "EVs"
    / "rh.txt"
)

PARAMETER_FILE = (
    PROJECT_ROOT
    / "configs"
    / "synthetic_ground_truth.json"
)

HRF_FILE = (
    PROJECT_ROOT
    / "data"
    / "synthetic"
    / "ground_truth_hrf.csv"
)

RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "synthetic"
    / "pinn_window_pilot"
)

FIGURES_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Datos:", NOISY_FILE.exists())
print("Eventos:", EVENT_FILE.exists())
print("Parámetros:", PARAMETER_FILE.exists())
print("HRF:", HRF_FILE.exists())

Datos: True
Eventos: True
Parámetros: True
HRF: True


In [4]:
data = pd.read_csv(
    NOISY_FILE
)

event_values = np.loadtxt(
    EVENT_FILE,
    ndmin=2,
)

events = pd.DataFrame(
    {
        "onset_s": event_values[:, 0],
        "duration_s": event_values[:, 1],
        "amplitude": event_values[:, 2],
    }
).sort_values(
    "onset_s"
).reset_index(
    drop=True
)

with open(
    PARAMETER_FILE,
    encoding="utf-8",
) as file:
    ground_truth_parameters = json.load(file)

ground_truth_hrf = pd.read_csv(
    HRF_FILE
)

time_full = data[
    "time_s"
].to_numpy(dtype=float)

noisy_full = data[
    "bold_noisy_fraction"
].to_numpy(dtype=float)

clean_full = data[
    "bold_clean_fraction"
].to_numpy(dtype=float)

test_mask = data[
    "test_mask"
].to_numpy(dtype=bool)

display(events)

,onset_s,duration_s,amplitude
0,11.009,12.0,1.0
1,131.892,12.0,1.0


In [5]:
first_event = events.iloc[0]

SECONDS_BEFORE = 2.0
SECONDS_AFTER = 20.0

window_start_s = (
    float(first_event["onset_s"])
    - SECONDS_BEFORE
)

window_end_s = (
    float(first_event["onset_s"])
    + float(first_event["duration_s"])
    + SECONDS_AFTER
)

window_mask = (
    (time_full >= window_start_s)
    & (time_full <= window_end_s)
)

time_window_absolute = time_full[
    window_mask
]

time_window = (
    time_window_absolute
    - window_start_s
)

noisy_window = noisy_full[
    window_mask
]

clean_window = clean_full[
    window_mask
]

stimulus_onset_relative = (
    float(first_event["onset_s"])
    - window_start_s
)

stimulus_duration_s = float(
    first_event["duration_s"]
)

stimulus_end_relative = (
    stimulus_onset_relative
    + stimulus_duration_s
)

print(
    f"Ventana absoluta: "
    f"{window_start_s:.3f}–{window_end_s:.3f} s"
)

print(
    f"Tiempo relativo: "
    f"{time_window[0]:.3f}–{time_window[-1]:.3f} s"
)

print(
    f"Estímulo relativo: "
    f"{stimulus_onset_relative:.3f}–"
    f"{stimulus_end_relative:.3f} s"
)

print("Puntos observados:", len(time_window))

Ventana absoluta: 9.009–43.009 s
Tiempo relativo: 0.351–33.471 s
Estímulo relativo: 2.000–14.000 s
Puntos observados: 47


In [6]:
RANDOM_SEED = 20260717

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )

torch.use_deterministic_algorithms(
    True,
    warn_only=True,
)

torch.set_default_dtype(
    torch.float64
)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Dispositivo:", DEVICE)
print("Precisión:", torch.get_default_dtype())

Dispositivo: cpu
Precisión: torch.float64


In [7]:
PINN_CONFIG = {
    "hidden_layers": [64, 64, 64],
    "phase_1_epochs": 1200,
    "phase_2_epochs": 3500,
    "phase_1_learning_rate": 1e-3,
    "phase_2_network_learning_rate": 3e-4,
    "phase_2_parameter_learning_rate": 8e-4,
    "lambda_data": 1.0,
    "lambda_physics_phase_1": 5.0,
    "lambda_physics_phase_2": 20.0,
    "lambda_initial": 100.0,
    "lambda_terminal": 5.0,
    "n_collocation": 700,
    "residual_scale": 0.05,
    "active_data_weight": 3.0,
    "lbfgs_max_iterations": 300,
}

INITIAL_PARAMETERS = {
    "epsilon": 0.12,
    "tau": 1.20,
    "alpha": 0.38,
}

PARAMETER_BOUNDS = {
    "epsilon": (0.05, 0.30),
    "tau": (0.50, 2.00),
    "alpha": (0.15, 0.60),
}

In [8]:
def logit(
    probability: float,
) -> float:
    probability = np.clip(
        probability,
        1e-8,
        1.0 - 1e-8,
    )

    return float(
        np.log(
            probability
            / (1.0 - probability)
        )
    )

In [9]:
class BoundedScalar(nn.Module):

    def __init__(
        self,
        lower: float,
        upper: float,
        initial: float,
    ):
        super().__init__()

        self.lower = float(lower)
        self.upper = float(upper)

        normalized = (
            (initial - lower)
            / (upper - lower)
        )

        self.raw = nn.Parameter(
            torch.tensor(
                logit(normalized),
                dtype=torch.float64,
            )
        )

    def forward(self):
        return (
            self.lower
            + (
                self.upper
                - self.lower
            )
            * torch.sigmoid(
                self.raw
            )
        )

In [10]:
class WindowPINN(nn.Module):

    def __init__(
        self,
        maximum_time_s: float,
    ):
        super().__init__()

        self.maximum_time_s = float(
            maximum_time_s
        )

        dimensions = [
            1,
            *PINN_CONFIG["hidden_layers"],
            4,
        ]

        layers = []

        for input_size, output_size in zip(
            dimensions[:-2],
            dimensions[1:-1],
        ):
            layers.append(
                nn.Linear(
                    input_size,
                    output_size,
                )
            )

            layers.append(
                nn.Tanh()
            )

        layers.append(
            nn.Linear(
                dimensions[-2],
                dimensions[-1],
            )
        )

        self.network = nn.Sequential(
            *layers
        )

        final_layer = self.network[-1]

        nn.init.zeros_(
            final_layer.weight
        )

        nn.init.zeros_(
            final_layer.bias
        )

        self.epsilon = BoundedScalar(
            *PARAMETER_BOUNDS["epsilon"],
            INITIAL_PARAMETERS["epsilon"],
        )

        self.tau = BoundedScalar(
            *PARAMETER_BOUNDS["tau"],
            INITIAL_PARAMETERS["tau"],
        )

        self.alpha = BoundedScalar(
            *PARAMETER_BOUNDS["alpha"],
            INITIAL_PARAMETERS["alpha"],
        )

    def states(
        self,
        time_tensor: torch.Tensor,
    ):
        normalized_time = (
            2.0
            * time_tensor
            / self.maximum_time_s
            - 1.0
        )

        raw = self.network(
            normalized_time
        )

        s = (
            0.50
            * torch.tanh(
                raw[:, 0:1]
            )
        )

        f = (
            1.0
            + 0.80
            * torch.tanh(
                raw[:, 1:2]
            )
        )

        v = (
            1.0
            + 0.35
            * torch.tanh(
                raw[:, 2:3]
            )
        )

        q = (
            1.0
            + 0.35
            * torch.tanh(
                raw[:, 3:4]
            )
        )

        return s, f, v, q

    def parameter_values(self):
        return {
            "epsilon": self.epsilon(),
            "tau": self.tau(),
            "alpha": self.alpha(),
        }

    def bold(
        self,
        time_tensor: torch.Tensor,
    ):
        _, _, v, q = self.states(
            time_tensor
        )

        E0 = 0.34
        V0 = 0.02

        k1 = 7.0 * E0
        k2 = 2.0
        k3 = 2.0 * E0 - 0.2

        return V0 * (
            k1 * (1.0 - q)
            + k2 * (
                1.0 - q / v
            )
            + k3 * (1.0 - v)
        )

In [11]:
def stimulus_torch(
    time_tensor: torch.Tensor,
) -> torch.Tensor:
    active = (
        (time_tensor >= stimulus_onset_relative)
        & (
            time_tensor
            < stimulus_end_relative
        )
    )

    return active.to(
        time_tensor.dtype
    )

In [12]:
def time_derivative(
    output: torch.Tensor,
    time_tensor: torch.Tensor,
):
    return torch.autograd.grad(
        outputs=output,
        inputs=time_tensor,
        grad_outputs=torch.ones_like(
            output
        ),
        create_graph=True,
        retain_graph=True,
    )[0]

In [13]:
def physical_residuals(
    model: WindowPINN,
    time_tensor: torch.Tensor,
):
    times = (
        time_tensor
        .detach()
        .clone()
        .requires_grad_(True)
    )

    s, f, v, q = model.states(
        times
    )

    ds_dt = time_derivative(
        s,
        times,
    )

    df_dt = time_derivative(
        f,
        times,
    )

    dv_dt = time_derivative(
        v,
        times,
    )

    dq_dt = time_derivative(
        q,
        times,
    )

    parameters = model.parameter_values()

    epsilon = parameters["epsilon"]
    tau = parameters["tau"]
    alpha = parameters["alpha"]

    kappa_s = 0.65
    kappa_f = 0.41
    E0 = 0.34

    neural_input = stimulus_torch(
        times
    )

    extraction = (
        1.0
        - torch.pow(
            torch.tensor(
                1.0 - E0,
                device=DEVICE,
            ),
            1.0 / f,
        )
    )

    residual_s = (
        ds_dt
        - (
            epsilon * neural_input
            - kappa_s * s
            - kappa_f * (f - 1.0)
        )
    )

    residual_f = (
        df_dt - s
    )

    residual_v = (
        tau * dv_dt
        - (
            f
            - torch.pow(
                v,
                1.0 / alpha,
            )
        )
    )

    residual_q = (
        tau * dq_dt
        - (
            f * extraction / E0
            - torch.pow(
                v,
                1.0 / alpha,
            )
            * q / v
        )
    )

    return {
        "s": residual_s,
        "f": residual_f,
        "v": residual_v,
        "q": residual_q,
    }

In [14]:
observed_time_tensor = torch.tensor(
    time_window[:, None],
    device=DEVICE,
)

observed_bold_tensor = torch.tensor(
    noisy_window[:, None],
    device=DEVICE,
)

data_scale = float(
    np.std(
        noisy_window,
        ddof=0,
    )
)

data_weights = np.ones(
    len(time_window),
    dtype=float,
)

active_response_mask = (
    (time_window >= stimulus_onset_relative)
    & (
        time_window
        <= stimulus_end_relative + 12.0
    )
)

data_weights[
    active_response_mask
] = PINN_CONFIG[
    "active_data_weight"
]

data_weight_tensor = torch.tensor(
    data_weights[:, None],
    device=DEVICE,
)

collocation_times = np.linspace(
    0.0,
    float(time_window[-1]),
    PINN_CONFIG["n_collocation"],
)

collocation_times = np.unique(
    np.concatenate(
        [
            collocation_times,
            time_window,
        ]
    )
)

for boundary in [
    stimulus_onset_relative,
    stimulus_end_relative,
]:
    collocation_times = collocation_times[
        np.abs(
            collocation_times - boundary
        ) > 0.05
    ]

collocation_tensor = torch.tensor(
    collocation_times[:, None],
    device=DEVICE,
)

print("Puntos observados:", len(time_window))
print("Puntos físicos:", len(collocation_times))
print("Escala BOLD:", data_scale)

Puntos observados: 47
Puntos físicos: 741
Escala BOLD: 0.006951117581788744


In [15]:
def calculate_losses(
    model: WindowPINN,
    lambda_physics: float,
):
    predicted_bold = model.bold(
        observed_time_tensor
    )

    squared_data_error = (
        (
            predicted_bold
            - observed_bold_tensor
        )
        / data_scale
    ) ** 2

    data_loss = torch.sum(
        data_weight_tensor
        * squared_data_error
    ) / torch.sum(
        data_weight_tensor
    )

    residuals = physical_residuals(
        model,
        collocation_tensor,
    )

    residual_scale = PINN_CONFIG[
        "residual_scale"
    ]

    physics_loss = sum(
        torch.mean(
            (
                residual
                / residual_scale
            ) ** 2
        )
        for residual in residuals.values()
    )

    initial_time = torch.zeros(
        (1, 1),
        device=DEVICE,
    )

    final_time = torch.full(
        (1, 1),
        float(time_window[-1]),
        device=DEVICE,
    )

    s0, f0, v0, q0 = model.states(
        initial_time
    )

    sf, ff, vf, qf = model.states(
        final_time
    )

    initial_loss = (
        torch.mean(s0**2)
        + torch.mean(
            (f0 - 1.0) ** 2
        )
        + torch.mean(
            (v0 - 1.0) ** 2
        )
        + torch.mean(
            (q0 - 1.0) ** 2
        )
    )

    terminal_loss = (
        torch.mean(sf**2)
        + torch.mean(
            (ff - 1.0) ** 2
        )
        + torch.mean(
            (vf - 1.0) ** 2
        )
        + torch.mean(
            (qf - 1.0) ** 2
        )
    )

    total_loss = (
        PINN_CONFIG["lambda_data"]
        * data_loss
        + lambda_physics
        * physics_loss
        + PINN_CONFIG["lambda_initial"]
        * initial_loss
        + PINN_CONFIG["lambda_terminal"]
        * terminal_loss
    )

    return {
        "total": total_loss,
        "data": data_loss,
        "physics": physics_loss,
        "initial": initial_loss,
        "terminal": terminal_loss,
    }

In [16]:
def train_adam_phase(
    model,
    optimizer,
    n_epochs,
    phase_name,
    lambda_physics,
):
    history = []

    best_loss = np.inf
    best_state = None

    start_time = time.time()

    for epoch in range(
        1,
        n_epochs + 1,
    ):
        model.train()
        optimizer.zero_grad()

        losses = calculate_losses(
            model,
            lambda_physics,
        )

        losses["total"].backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0,
        )

        optimizer.step()

        total_value = float(
            losses["total"]
            .detach()
            .cpu()
        )

        parameters = {
            name: float(
                value.detach().cpu()
            )
            for name, value in (
                model.parameter_values()
            ).items()
        }

        history.append(
            {
                "phase": phase_name,
                "epoch": epoch,
                "total_loss": total_value,
                "data_loss": float(
                    losses["data"]
                    .detach()
                    .cpu()
                ),
                "physics_loss": float(
                    losses["physics"]
                    .detach()
                    .cpu()
                ),
                "initial_loss": float(
                    losses["initial"]
                    .detach()
                    .cpu()
                ),
                "terminal_loss": float(
                    losses["terminal"]
                    .detach()
                    .cpu()
                ),
                **parameters,
            }
        )

        if total_value < best_loss:
            best_loss = total_value
            best_state = copy.deepcopy(
                model.state_dict()
            )

        if (
            epoch == 1
            or epoch % 300 == 0
            or epoch == n_epochs
        ):
            print(
                f"{phase_name} | "
                f"{epoch}/{n_epochs} | "
                f"L={total_value:.5f} | "
                f"Ldata={history[-1]['data_loss']:.5f} | "
                f"Lphys={history[-1]['physics_loss']:.5f} | "
                f"epsilon={parameters['epsilon']:.4f} | "
                f"tau={parameters['tau']:.4f} | "
                f"alpha={parameters['alpha']:.4f}"
            )

    model.load_state_dict(
        best_state
    )

    print(
        f"{phase_name} finalizada en "
        f"{(time.time() - start_time) / 60:.2f} min"
    )

    return pd.DataFrame(history)

In [17]:
NOISE_SUMMARY_PATH = (
    PROJECT_ROOT
    / "results"
    / "synthetic"
    / "noise_dataset_summary.csv"
)

GROUND_TRUTH_HRF_PATH = (
    PROJECT_ROOT
    / "data"
    / "synthetic"
    / "ground_truth_hrf.csv"
)

NOISY_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "synthetic"
    / "noisy"
)

BATCH_RESULTS_DIR = (
    PROJECT_ROOT
    / "results"
    / "synthetic"
    / "pinn"
)

PREDICTIONS_DIR = (
    BATCH_RESULTS_DIR
    / "predictions"
)

HRF_ESTIMATES_DIR = (
    BATCH_RESULTS_DIR
    / "hrf_estimates"
)

HISTORIES_DIR = (
    BATCH_RESULTS_DIR
    / "histories"
)

PARAMETERS_DIR = (
    BATCH_RESULTS_DIR
    / "parameters"
)

for directory in [
    BATCH_RESULTS_DIR,
    PREDICTIONS_DIR,
    HRF_ESTIMATES_DIR,
    HISTORIES_DIR,
    PARAMETERS_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

noise_summary = pd.read_csv(
    NOISE_SUMMARY_PATH
)

ground_truth_hrf = pd.read_csv(
    GROUND_TRUTH_HRF_PATH
)

print("Experimentos disponibles:", len(noise_summary))

Experimentos disponibles: 60


In [18]:
SCENARIO_EVENT_FILES = {
    "motor_lr_m1_left": (
        PROJECT_ROOT
        / "data"
        / "raw"
        / "100206"
        / "MNINonLinear"
        / "Results"
        / "tfMRI_MOTOR_LR"
        / "EVs"
        / "rh.txt"
    ),
    "motor_lr_m1_right": (
        PROJECT_ROOT
        / "data"
        / "raw"
        / "100206"
        / "MNINonLinear"
        / "Results"
        / "tfMRI_MOTOR_LR"
        / "EVs"
        / "lh.txt"
    ),
    "motor_rl_m1_left": (
        PROJECT_ROOT
        / "data"
        / "raw"
        / "100206"
        / "MNINonLinear"
        / "Results"
        / "tfMRI_MOTOR_RL"
        / "EVs"
        / "rh.txt"
    ),
    "motor_rl_m1_right": (
        PROJECT_ROOT
        / "data"
        / "raw"
        / "100206"
        / "MNINonLinear"
        / "Results"
        / "tfMRI_MOTOR_RL"
        / "EVs"
        / "lh.txt"
    ),
}

for scenario_name, event_path in (
    SCENARIO_EVENT_FILES.items()
):
    print(
        scenario_name,
        event_path.exists(),
    )

motor_lr_m1_left True
motor_lr_m1_right True
motor_rl_m1_left True
motor_rl_m1_right True


In [19]:
def preparar_experimento(
    experiment: pd.Series,
) -> dict:
    global stimulus_onset_relative
    global stimulus_end_relative
    global observed_time_tensor
    global observed_bold_tensor
    global data_scale
    global data_weight_tensor
    global collocation_tensor

    scenario_name = str(
        experiment["scenario"]
    )

    snr_name = str(
        experiment["snr_name"]
    )

    replicate = int(
        experiment["replicate"]
    )

    file_name = (
        f"{scenario_name}_"
        f"{snr_name}_"
        f"rep_{replicate:02d}.csv"
    )

    noisy_path = (
        NOISY_DATA_DIR
        / scenario_name
        / snr_name
        / file_name
    )

    if not noisy_path.exists():
        raise FileNotFoundError(
            noisy_path
        )

    table = pd.read_csv(
        noisy_path
    )

    event_path = (
        SCENARIO_EVENT_FILES[
            scenario_name
        ]
    )

    event_values = np.loadtxt(
        event_path,
        ndmin=2,
    )

    experiment_events = pd.DataFrame(
        {
            "onset_s": event_values[:, 0],
            "duration_s": event_values[:, 1],
            "amplitude": event_values[:, 2],
        }
    ).sort_values(
        "onset_s"
    ).reset_index(
        drop=True
    )

    time_full = table[
        "time_s"
    ].to_numpy(dtype=float)

    noisy_full = table[
        "bold_noisy_fraction"
    ].to_numpy(dtype=float)

    clean_full = table[
        "bold_clean_fraction"
    ].to_numpy(dtype=float)

    test_mask_local = table[
        "test_mask"
    ].to_numpy(dtype=bool)

    first_event = experiment_events.iloc[0]

    window_start_s = (
        float(first_event["onset_s"])
        - SECONDS_BEFORE
    )

    window_end_s = (
        float(first_event["onset_s"])
        + float(first_event["duration_s"])
        + SECONDS_AFTER
    )

    window_mask = (
        (time_full >= window_start_s)
        & (time_full <= window_end_s)
    )

    time_window_local = (
        time_full[window_mask]
        - window_start_s
    )

    noisy_window_local = (
        noisy_full[window_mask]
    )

    stimulus_onset_relative = (
        float(first_event["onset_s"])
        - window_start_s
    )

    stimulus_end_relative = (
        stimulus_onset_relative
        + float(first_event["duration_s"])
    )

    observed_time_tensor = torch.tensor(
        time_window_local[:, None],
        device=DEVICE,
    )

    observed_bold_tensor = torch.tensor(
        noisy_window_local[:, None],
        device=DEVICE,
    )

    data_scale = float(
        np.std(
            noisy_window_local,
            ddof=0,
        )
    )

    data_weights = np.ones(
        len(time_window_local),
        dtype=float,
    )

    active_response_mask = (
        (
            time_window_local
            >= stimulus_onset_relative
        )
        & (
            time_window_local
            <= stimulus_end_relative + 12.0
        )
    )

    data_weights[
        active_response_mask
    ] = PINN_CONFIG[
        "active_data_weight"
    ]

    data_weight_tensor = torch.tensor(
        data_weights[:, None],
        device=DEVICE,
    )

    collocation_times = np.linspace(
        0.0,
        float(time_window_local[-1]),
        PINN_CONFIG["n_collocation"],
    )

    collocation_times = np.unique(
        np.concatenate(
            [
                collocation_times,
                time_window_local,
            ]
        )
    )

    for boundary in [
        stimulus_onset_relative,
        stimulus_end_relative,
    ]:
        collocation_times = (
            collocation_times[
                np.abs(
                    collocation_times
                    - boundary
                ) > 0.05
            ]
        )

    collocation_tensor = torch.tensor(
        collocation_times[:, None],
        device=DEVICE,
    )

    return {
        "scenario": scenario_name,
        "snr_name": snr_name,
        "replicate": replicate,
        "table": table,
        "events": experiment_events,
        "time_full": time_full,
        "noisy_full": noisy_full,
        "clean_full": clean_full,
        "test_mask": test_mask_local,
        "time_window": time_window_local,
        "maximum_window_time": float(
            time_window_local[-1]
        ),
    }

In [20]:
def entrenar_fase_batch(
    model,
    optimizer,
    n_epochs,
    phase_name,
    lambda_physics,
):
    history = []

    best_loss = np.inf
    best_state = None

    for epoch in range(
        1,
        n_epochs + 1,
    ):
        model.train()
        optimizer.zero_grad()

        losses = calculate_losses(
            model,
            lambda_physics,
        )

        if not torch.isfinite(
            losses["total"]
        ):
            raise RuntimeError(
                "Se obtuvo una pérdida no finita."
            )

        losses["total"].backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0,
        )

        optimizer.step()

        total_value = float(
            losses["total"]
            .detach()
            .cpu()
        )

        parameter_values = {
            name: float(
                value.detach().cpu()
            )
            for name, value in (
                model.parameter_values()
            ).items()
        }

        history.append(
            {
                "phase": phase_name,
                "epoch": epoch,
                "total_loss": total_value,
                "data_loss": float(
                    losses["data"]
                    .detach()
                    .cpu()
                ),
                "physics_loss": float(
                    losses["physics"]
                    .detach()
                    .cpu()
                ),
                "initial_loss": float(
                    losses["initial"]
                    .detach()
                    .cpu()
                ),
                "terminal_loss": float(
                    losses["terminal"]
                    .detach()
                    .cpu()
                ),
                **parameter_values,
            }
        )

        if total_value < best_loss:
            best_loss = total_value

            best_state = copy.deepcopy(
                model.state_dict()
            )

    if best_state is None:
        raise RuntimeError(
            "No se guardó un modelo válido."
        )

    model.load_state_dict(
        best_state
    )

    return pd.DataFrame(
        history
    )

In [21]:
def simular_modelo_ode(
    evaluation_times: np.ndarray,
    experiment_events: pd.DataFrame,
    estimated_parameters: dict,
) -> pd.DataFrame:
    parameters = {
        **ground_truth_parameters,
        **estimated_parameters,
    }

    onsets = (
        experiment_events["onset_s"]
        .to_numpy(dtype=float)
    )

    durations = (
        experiment_events["duration_s"]
        .to_numpy(dtype=float)
    )

    amplitudes = (
        experiment_events["amplitude"]
        .to_numpy(dtype=float)
    )

    def stimulus(current_time):
        active = (
            (current_time >= onsets)
            & (
                current_time
                < onsets + durations
            )
        )

        return float(
            np.sum(
                amplitudes[active]
            )
        )

    def rhs(
        current_time,
        state,
    ):
        s, f, v, q = state

        epsilon = parameters["epsilon"]
        tau = parameters["tau"]
        alpha = parameters["alpha"]
        kappa_s = parameters["kappa_s"]
        kappa_f = parameters["kappa_f"]
        E0 = parameters["E0"]

        f_safe = max(
            float(f),
            1e-8,
        )

        v_safe = max(
            float(v),
            1e-8,
        )

        extraction = (
            1.0
            - (1.0 - E0) ** (
                1.0 / f_safe
            )
        )

        neural_input = stimulus(
            current_time
        )

        return [
            (
                epsilon * neural_input
                - kappa_s * s
                - kappa_f * (f - 1.0)
            ),
            s,
            (
                f
                - v_safe ** (
                    1.0 / alpha
                )
            ) / tau,
            (
                f * extraction / E0
                - v_safe ** (
                    1.0 / alpha
                )
                * q / v_safe
            ) / tau,
        ]

    solution = solve_ivp(
        fun=rhs,
        t_span=(
            float(evaluation_times[0]),
            float(evaluation_times[-1]),
        ),
        y0=[
            0.0,
            1.0,
            1.0,
            1.0,
        ],
        t_eval=evaluation_times,
        max_step=0.02,
        rtol=1e-9,
        atol=1e-11,
    )

    if not solution.success:
        raise RuntimeError(
            solution.message
        )

    s, f, v, q = solution.y

    V0 = parameters["V0"]
    k1 = parameters["k1"]
    k2 = parameters["k2"]
    k3 = parameters["k3"]

    bold = V0 * (
        k1 * (1.0 - q)
        + k2 * (
            1.0 - q / v
        )
        + k3 * (
            1.0 - v
        )
    )

    return pd.DataFrame(
        {
            "time_s": evaluation_times,
            "s": s,
            "f": f,
            "v": v,
            "q": q,
            "bold_fraction": bold,
        }
    )

In [22]:
hrf_reference = ground_truth_hrf.loc[
    ground_truth_hrf[
        "time_from_onset_s"
    ] >= 0.0
].copy()

HRF_EVALUATION_TIMES = (
    hrf_reference[
        "time_from_onset_s"
    ].to_numpy(dtype=float)
)

HRF_TRUE = (
    hrf_reference[
        "bold_fraction"
    ].to_numpy(dtype=float)
)

In [23]:
def estimar_hrf_fisiologica(
    estimated_parameters: dict,
) -> np.ndarray:
    absolute_times = (
        HRF_EVALUATION_TIMES
        + 1.0
    )

    hrf_event = pd.DataFrame(
        {
            "onset_s": [1.0],
            "duration_s": [1.0],
            "amplitude": [1.0],
        }
    )

    simulation = simular_modelo_ode(
        evaluation_times=absolute_times,
        experiment_events=hrf_event,
        estimated_parameters=estimated_parameters,
    )

    return simulation[
        "bold_fraction"
    ].to_numpy(dtype=float)

In [24]:
def correlacion_segura(
    observed,
    predicted,
):
    if (
        np.std(observed) < 1e-12
        or np.std(predicted) < 1e-12
    ):
        return np.nan

    return float(
        np.corrcoef(
            observed,
            predicted,
        )[0, 1]
    )

In [25]:
def calcular_metricas_batch(
    observed,
    predicted,
):
    mse = mean_squared_error(
        observed,
        predicted,
    )

    return {
        "rmse_percent": float(
            100.0 * np.sqrt(mse)
        ),
        "mae_percent": float(
            100.0
            * mean_absolute_error(
                observed,
                predicted,
            )
        ),
        "r2": float(
            r2_score(
                observed,
                predicted,
            )
        ),
        "pearson_r": correlacion_segura(
            observed,
            predicted,
        ),
    }

In [26]:
def calcular_metricas_hrf_batch(
    true_hrf,
    estimated_hrf,
):
    metrics = calcular_metricas_batch(
        true_hrf,
        estimated_hrf,
    )

    true_peak_index = int(
        np.argmax(true_hrf)
    )

    estimated_peak_index = int(
        np.argmax(estimated_hrf)
    )

    metrics.update(
        {
            "true_peak_percent": float(
                100.0
                * true_hrf[
                    true_peak_index
                ]
            ),
            "estimated_peak_percent": float(
                100.0
                * estimated_hrf[
                    estimated_peak_index
                ]
            ),
            "peak_error_percent": float(
                100.0
                * (
                    estimated_hrf[
                        estimated_peak_index
                    ]
                    - true_hrf[
                        true_peak_index
                    ]
                )
            ),
            "true_time_to_peak_s": float(
                HRF_EVALUATION_TIMES[
                    true_peak_index
                ]
            ),
            "estimated_time_to_peak_s": float(
                HRF_EVALUATION_TIMES[
                    estimated_peak_index
                ]
            ),
        }
    )

    metrics[
        "time_to_peak_error_s"
    ] = (
        metrics[
            "estimated_time_to_peak_s"
        ]
        - metrics[
            "true_time_to_peak_s"
        ]
    )

    return metrics

In [27]:
def ejecutar_experimento_pinn(
    experiment: pd.Series,
) -> dict:
    context = preparar_experimento(
        experiment
    )

    model_seed = (
        20260717
        + int(
            experiment["random_seed"]
        )
    )

    random.seed(model_seed)
    np.random.seed(model_seed)
    torch.manual_seed(model_seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            model_seed
        )

    model = WindowPINN(
        maximum_time_s=context[
            "maximum_window_time"
        ]
    ).to(DEVICE)

    start_time = time.time()

    # Fase 1
    model.epsilon.raw.requires_grad_(
        False
    )

    model.tau.raw.requires_grad_(
        False
    )

    model.alpha.raw.requires_grad_(
        False
    )

    optimizer_phase_1 = torch.optim.Adam(
        model.network.parameters(),
        lr=PINN_CONFIG[
            "phase_1_learning_rate"
        ],
    )

    history_phase_1 = (
        entrenar_fase_batch(
            model=model,
            optimizer=optimizer_phase_1,
            n_epochs=PINN_CONFIG[
                "phase_1_epochs"
            ],
            phase_name="network_warmup",
            lambda_physics=PINN_CONFIG[
                "lambda_physics_phase_1"
            ],
        )
    )

    # Fase 2
    model.epsilon.raw.requires_grad_(
        True
    )

    model.tau.raw.requires_grad_(
        True
    )

    model.alpha.raw.requires_grad_(
        True
    )

    optimizer_phase_2 = torch.optim.Adam(
        [
            {
                "params": (
                    model.network.parameters()
                ),
                "lr": PINN_CONFIG[
                    "phase_2_network_learning_rate"
                ],
            },
            {
                "params": [
                    model.epsilon.raw,
                    model.tau.raw,
                    model.alpha.raw,
                ],
                "lr": PINN_CONFIG[
                    "phase_2_parameter_learning_rate"
                ],
            },
        ]
    )

    history_phase_2 = (
        entrenar_fase_batch(
            model=model,
            optimizer=optimizer_phase_2,
            n_epochs=PINN_CONFIG[
                "phase_2_epochs"
            ],
            phase_name="inverse_problem",
            lambda_physics=PINN_CONFIG[
                "lambda_physics_phase_2"
            ],
        )
    )

    # L-BFGS
    lbfgs = torch.optim.LBFGS(
        model.parameters(),
        lr=0.5,
        max_iter=PINN_CONFIG[
            "lbfgs_max_iterations"
        ],
        history_size=50,
        line_search_fn="strong_wolfe",
    )

    def closure():
        lbfgs.zero_grad()

        losses = calculate_losses(
            model,
            PINN_CONFIG[
                "lambda_physics_phase_2"
            ],
        )

        losses["total"].backward()

        return losses["total"]

    _ = lbfgs.step(
        closure
    )

    estimated_parameters = {
        name: float(
            value.detach().cpu()
        )
        for name, value in (
            model.parameter_values()
        ).items()
    }

    forward_simulation = (
        simular_modelo_ode(
            evaluation_times=context[
                "time_full"
            ],
            experiment_events=context[
                "events"
            ],
            estimated_parameters=(
                estimated_parameters
            ),
        )
    )

    forward_bold = (
        forward_simulation[
            "bold_fraction"
        ].to_numpy(dtype=float)
    )

    test_metrics = (
        calcular_metricas_batch(
            context["clean_full"][
                context["test_mask"]
            ],
            forward_bold[
                context["test_mask"]
            ],
        )
    )

    estimated_hrf = (
        estimar_hrf_fisiologica(
            estimated_parameters
        )
    )

    hrf_metrics = (
        calcular_metricas_hrf_batch(
            HRF_TRUE,
            estimated_hrf,
        )
    )

    elapsed_minutes = (
        time.time() - start_time
    ) / 60.0

    history = pd.concat(
        [
            history_phase_1,
            history_phase_2,
        ],
        ignore_index=True,
    )

    return {
        "context": context,
        "model": model,
        "estimated_parameters": (
            estimated_parameters
        ),
        "forward_simulation": (
            forward_simulation
        ),
        "forward_bold": forward_bold,
        "test_metrics": test_metrics,
        "estimated_hrf": estimated_hrf,
        "hrf_metrics": hrf_metrics,
        "history": history,
        "elapsed_minutes": (
            elapsed_minutes
        ),
        "model_seed": model_seed,
    }

In [28]:
METRICS_PATH = (
    BATCH_RESULTS_DIR
    / "pinn_synthetic_metrics.csv"
)

if METRICS_PATH.exists():
    completed_metrics = pd.read_csv(
        METRICS_PATH
    )
else:
    completed_metrics = pd.DataFrame()

completed_keys = set()

if not completed_metrics.empty:
    completed_keys = set(
        zip(
            completed_metrics["scenario"],
            completed_metrics["snr_name"],
            completed_metrics[
                "replicate"
            ].astype(int),
        )
    )

print(
    "Experimentos ya terminados:",
    len(completed_keys),
)

Experimentos ya terminados: 0


In [30]:
for experiment_index, (_, experiment) in enumerate(
    noise_summary.iterrows(),
    start=1,
):
    key = (
        str(experiment["scenario"]),
        str(experiment["snr_name"]),
        int(experiment["replicate"]),
    )

    if key in completed_keys:
        print(
            f"Omitido {key}: ya existe."
        )
        continue

    print(
        "\n"
        + "=" * 70
    )

    print(
        f"Experimento "
        f"{experiment_index}/"
        f"{len(noise_summary)}: "
        f"{key}"
    )

    result = ejecutar_experimento_pinn(
        experiment
    )

    context = result["context"]

    metrics_row = {
        "scenario": key[0],
        "run": experiment["run"],
        "roi": experiment["roi"],
        "condition": experiment[
            "condition"
        ],
        "snr_name": key[1],
        "target_snr": float(
            experiment["target_snr"]
        ),
        "replicate": key[2],
        "random_seed": result[
            "model_seed"
        ],
        "epsilon_estimated": result[
            "estimated_parameters"
        ]["epsilon"],
        "tau_estimated": result[
            "estimated_parameters"
        ]["tau"],
        "alpha_estimated": result[
            "estimated_parameters"
        ]["alpha"],
        "epsilon_relative_error_percent": (
            100.0
            * abs(
                result[
                    "estimated_parameters"
                ]["epsilon"]
                - ground_truth_parameters[
                    "epsilon"
                ]
            )
            / ground_truth_parameters[
                "epsilon"
            ]
        ),
        "tau_relative_error_percent": (
            100.0
            * abs(
                result[
                    "estimated_parameters"
                ]["tau"]
                - ground_truth_parameters[
                    "tau"
                ]
            )
            / ground_truth_parameters[
                "tau"
            ]
        ),
        "alpha_relative_error_percent": (
            100.0
            * abs(
                result[
                    "estimated_parameters"
                ]["alpha"]
                - ground_truth_parameters[
                    "alpha"
                ]
            )
            / ground_truth_parameters[
                "alpha"
            ]
        ),
        "test_rmse_percent": result[
            "test_metrics"
        ]["rmse_percent"],
        "test_mae_percent": result[
            "test_metrics"
        ]["mae_percent"],
        "test_r2": result[
            "test_metrics"
        ]["r2"],
        "test_pearson_r": result[
            "test_metrics"
        ]["pearson_r"],
        "hrf_rmse_percent": result[
            "hrf_metrics"
        ]["rmse_percent"],
        "hrf_r2": result[
            "hrf_metrics"
        ]["r2"],
        "hrf_pearson_r": result[
            "hrf_metrics"
        ]["pearson_r"],
        "hrf_peak_error_percent": result[
            "hrf_metrics"
        ]["peak_error_percent"],
        "hrf_time_to_peak_error_s": result[
            "hrf_metrics"
        ]["time_to_peak_error_s"],
        "elapsed_minutes": result[
            "elapsed_minutes"
        ],
    }

    completed_metrics = pd.concat(
        [
            completed_metrics,
            pd.DataFrame(
                [metrics_row]
            ),
        ],
        ignore_index=True,
    )

    completed_metrics.to_csv(
        METRICS_PATH,
        index=False,
    )

    file_key = (
        f"{key[0]}_"
        f"{key[1]}_"
        f"rep_{key[2]:02d}"
    )

    prediction_table = (
        result[
            "forward_simulation"
        ].copy()
    )

    prediction_table[
        "clean_fraction"
    ] = context["clean_full"]

    prediction_table[
        "noisy_fraction"
    ] = context["noisy_full"]

    prediction_table[
        "test_mask"
    ] = context[
        "test_mask"
    ].astype(int)

    prediction_table.to_csv(
        PREDICTIONS_DIR
        / f"{file_key}.csv",
        index=False,
    )

    pd.DataFrame(
        {
            "time_from_onset_s": (
                HRF_EVALUATION_TIMES
            ),
            "true_hrf_fraction": HRF_TRUE,
            "estimated_hrf_fraction": (
                result["estimated_hrf"]
            ),
        }
    ).to_csv(
        HRF_ESTIMATES_DIR
        / f"{file_key}.csv",
        index=False,
    )

    result["history"].to_csv(
        HISTORIES_DIR
        / f"{file_key}.csv",
        index=False,
    )

    pd.DataFrame(
        [
            result[
                "estimated_parameters"
            ]
        ]
    ).to_csv(
        PARAMETERS_DIR
        / f"{file_key}.csv",
        index=False,
    )

    completed_keys.add(key)

    print(
        f"Terminado en "
        f"{result['elapsed_minutes']:.2f} min | "
        f"RMSE={metrics_row['test_rmse_percent']:.4f}% | "
        f"R2={metrics_row['test_r2']:.4f}"
    )

    del result

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Omitido ('motor_lr_m1_left', 'snr_10', 1): ya existe.
Omitido ('motor_lr_m1_left', 'snr_10', 2): ya existe.
Omitido ('motor_lr_m1_left', 'snr_10', 3): ya existe.
Omitido ('motor_lr_m1_left', 'snr_10', 4): ya existe.
Omitido ('motor_lr_m1_left', 'snr_10', 5): ya existe.
Omitido ('motor_lr_m1_left', 'snr_5', 1): ya existe.
Omitido ('motor_lr_m1_left', 'snr_5', 2): ya existe.
Omitido ('motor_lr_m1_left', 'snr_5', 3): ya existe.
Omitido ('motor_lr_m1_left', 'snr_5', 4): ya existe.
Omitido ('motor_lr_m1_left', 'snr_5', 5): ya existe.
Omitido ('motor_lr_m1_left', 'snr_2', 1): ya existe.
Omitido ('motor_lr_m1_left', 'snr_2', 2): ya existe.
Omitido ('motor_lr_m1_left', 'snr_2', 3): ya existe.
Omitido ('motor_lr_m1_left', 'snr_2', 4): ya existe.
Omitido ('motor_lr_m1_left', 'snr_2', 5): ya existe.
Omitido ('motor_lr_m1_right', 'snr_10', 1): ya existe.
Omitido ('motor_lr_m1_right', 'snr_10', 2): ya existe.
Omitido ('motor_lr_m1_right', 'snr_10', 3): ya existe.
Omitido ('motor_lr_m1_right', 'snr_

In [31]:
pinn_metrics = pd.read_csv(
    METRICS_PATH
)

print(
    "Experimentos completados:",
    len(pinn_metrics),
)

Experimentos completados: 60


In [32]:
pinn_grouped_summary = (
    pinn_metrics
    .groupby(
        [
            "snr_name",
            "target_snr",
        ],
        as_index=False,
    )
    .agg(
        n_experiments=(
            "test_rmse_percent",
            "size",
        ),
        test_rmse_mean=(
            "test_rmse_percent",
            "mean",
        ),
        test_rmse_std=(
            "test_rmse_percent",
            "std",
        ),
        test_r2_mean=(
            "test_r2",
            "mean",
        ),
        test_r2_std=(
            "test_r2",
            "std",
        ),
        test_pearson_mean=(
            "test_pearson_r",
            "mean",
        ),
        hrf_rmse_mean=(
            "hrf_rmse_percent",
            "mean",
        ),
        hrf_r2_mean=(
            "hrf_r2",
            "mean",
        ),
        hrf_pearson_mean=(
            "hrf_pearson_r",
            "mean",
        ),
        epsilon_error_mean=(
            "epsilon_relative_error_percent",
            "mean",
        ),
        tau_error_mean=(
            "tau_relative_error_percent",
            "mean",
        ),
        alpha_error_mean=(
            "alpha_relative_error_percent",
            "mean",
        ),
        elapsed_minutes_mean=(
            "elapsed_minutes",
            "mean",
        ),
    )
    .sort_values(
        "target_snr",
        ascending=False,
    )
)

pinn_grouped_summary.to_csv(
    BATCH_RESULTS_DIR
    / "pinn_synthetic_grouped_summary.csv",
    index=False,
)

display(
    pinn_grouped_summary
)

,snr_name,target_snr,n_experiments,test_rmse_mean,test_rmse_std,test_r2_mean,test_r2_std,test_pearson_mean,hrf_rmse_mean,hrf_r2_mean,hrf_pearson_mean,epsilon_error_mean,tau_error_mean,alpha_error_mean,elapsed_minutes_mean
0,snr_10,10.0,20,0.057740,0.023428,0.992228,0.006462,0.999591,0.012376,0.992284,0.998578,7.146198,10.514325,2.844672,2.088964
2,snr_5,5.0,20,0.065453,0.026332,0.990036,0.007621,0.999569,0.013915,0.990194,0.998496,8.701173,9.876182,2.463967,2.087607
1,snr_2,2.0,20,0.079970,0.042812,0.983597,0.017170,0.999121,0.018141,0.982553,0.997008,10.472877,14.259542,2.745069,2.092683
